## Load libraries

In [25]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, Optional
from datetime import datetime
from dotenv import load_dotenv
from cryptography.fernet import Fernet
from huggingface_hub import InferenceClient
from langchain_huggingface import HuggingFaceEndpoint
from langchain_groq import ChatGroq

import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd().parent
PRIMARY_DATA_PATH = BASE_DIR / "app" / "data" / "cleaned" / "sql_supermarket.parquet"
CUSTOMER_METRICS_PATH = BASE_DIR / "app" / "data" / "customer_day_metrics" / "customer_day_metrics.parquet"

# Load environment variables
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

print(f"✅ All libraries loaded at {datetime.now()}")

# Initialize security
SECURITY_KEY = Fernet.generate_key()
cipher = Fernet(SECURITY_KEY)
print(f"🔐 Security key generated: {SECURITY_KEY[:16]}...")

# Configure API Keys from environment or input
HUGGINGFACE_API_KEY = os.getenv('HUGGINGFACE_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

print(f"🔑 API Keys configured:")
print(f"  - HuggingFace: {'✅' if HUGGINGFACE_API_KEY else '❌'}")
print(f"  - Groq: {'✅' if GROQ_API_KEY else '❌'}")

✅ All libraries loaded at 2026-07-05 19:51:12.078979
🔐 Security key generated: b'TYqBTx7dPgzmSYgy'...
🔑 API Keys configured:
  - HuggingFace: ✅
  - Groq: ✅


## Load database

In [26]:
primary_df = pd.read_parquet(PRIMARY_DATA_PATH)
customer_day_metrics = pd.read_parquet(CUSTOMER_METRICS_PATH)

primary_df['order_date'] = pd.to_datetime(primary_df['order_date'], errors='coerce')
customer_day_metrics['order_date'] = pd.to_datetime(customer_day_metrics['order_date'], errors='coerce')

In [27]:
primary_df.head(2)

,id,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,...,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,unit_price,profit_margin
0,1,AG-2011-2040,2011-01-01,2011-06-01,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,...,"Tenex Lockers, Blue",4080000.0,2,0.0,1061400.0,354600.0,Medium,2011,2040000.0,0.26
1,2,IN-2011-47883,2011-01-01,2011-08-01,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,...,"Acme Trimmer, High Speed",1200000.0,3,0.1,360360.0,97200.0,Medium,2011,400000.0,0.30


In [28]:
customer_day_metrics.head(2)

,customer_name,order_date,daily_orders,weekly_orders,monthly_orders,spike_ratio,velocity_alert_flag
0,Aaron Bergman,2011-03-11,1,1.0,1.0,0.0,0
1,Aaron Bergman,2011-04-04,1,2.0,2.0,0.0,0


## Multi LLM Manager with Failover Support

In [29]:
from huggingface_hub import InferenceClient
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Create a class for multi LLM to easy get use for HuggingFace and Groq LLMs
class MultiLLM:
    def __init__(self, hf_api_key: str, groq_api_key: str):
        self.hf_api_key = hf_api_key
        self.groq_api_key = groq_api_key

    def huggingface_endpoint(self, max_new_tokens: int = 256) -> HuggingFaceEndpoint:
        """Returns a LangChain HuggingFaceEndpoint instance."""
        llm = HuggingFaceEndpoint(
            repo_id="meta-llama/Llama-3.1-8B-Instruct", 
            task="conversational",
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            huggingfacehub_api_token=self.hf_api_key,
            timeout=30,
        )

        chat_llm = ChatHuggingFace(llm=llm)
        return chat_llm
    
    def groq_endpoint(self, max_tokens: int = 256) -> ChatGroq:
        """Returns a LangChain ChatGroq instance."""
        llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.3,
            max_tokens=max_tokens,
            groq_api_key=self.groq_api_key
        )
        return llm

# Call the MultiLLM class to initialize both LLMs
multi_llm = MultiLLM(hf_api_key=HUGGINGFACE_API_KEY, groq_api_key=GROQ_API_KEY)

# Fetch plug and play models directly for agents
hf_model = multi_llm.huggingface_endpoint()
groq_model = multi_llm.groq_endpoint()

## define the abuse agent

In [30]:
from typing import Dict, Any, Optional
import pandas as pd
import numpy as np

class AbuseInvestigationAgent:
    def __init__(self, hf_llm=None, groq_llm=None, historical_df: pd.DataFrame = None):
        self.hf_llm = hf_llm
        self.groq_llm = groq_llm
        self.historical_data = historical_df.copy() if historical_df is not None else pd.DataFrame()

        if not self.historical_data.empty:
            self.historical_data["order_date"] = pd.to_datetime(self.historical_data["order_date"], errors="coerce")

    def _get_llm(self):
        if self.hf_llm is not None:
            return self.hf_llm
        if self.groq_llm is not None:
            return self.groq_llm
        return None

    def _call_llm_with_fallback(self, prompt: str) -> str:
        errors = []

        from langchain_core.messages import HumanMessage
        messages = [HumanMessage(content=prompt)]

        if self.hf_llm is not None:
            try:
                response = self.hf_llm.invoke(messages)
                return response.content # Extract text content
            except Exception as e:
                errors.append(f"HuggingFace failed: {e}")

        if self.groq_llm is not None:
            try:
                # Invoke using structured messages
                response = self.groq_llm.invoke(messages)
                return response.content 
            except Exception as e:
                errors.append(f"Groq failed: {e}")

        return f"LLM unavailable. Errors: {errors}"

    def _get_customer_day_row(self, customer_name: str, order_date) -> Optional[pd.Series]:
        if self.historical_data.empty or not customer_name or pd.isna(order_date):
            return None

        target_date = pd.to_datetime(order_date).normalize()
        data = self.historical_data.copy()
        data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce").dt.normalize()

        exact_match = data[
            (data["customer_name"] == customer_name) &
            (data["order_date"] == target_date)
        ]

        if not exact_match.empty:
            return exact_match.iloc[0]

        customer_only = data[data["customer_name"] == customer_name].copy()
        if customer_only.empty:
            return None

        customer_only["distance"] = (customer_only["order_date"] - target_date).abs()
        return customer_only.sort_values("distance").iloc[0]

    def _categorize_risk(self, abuse_score: float) -> str:
        if abuse_score >= 0.8:
            return "CRITICAL"
        elif abuse_score >= 0.6:
            return "HIGH"
        elif abuse_score >= 0.4:
            return "MEDIUM"
        return "LOW"

    def _analyze_pattern(self, metrics_row: Optional[pd.Series], abuse_score: float) -> list:
        patterns = []
        if metrics_row is None:
            return patterns

        if int(metrics_row.get("velocity_alert_flag", 0)) == 1:
            patterns.append("Velocity Alert Active")
        if float(metrics_row.get("daily_orders", 0)) >= 5:
            patterns.append("High Daily Transaction Velocity")
        if float(metrics_row.get("spike_ratio", 0)) >= 2.0:
            patterns.append("Spike Above Baseline")
        if float(metrics_row.get("monthly_orders", 0)) >= 60:
            patterns.append("High Monthly Transaction Volume")
        if abuse_score >= 0.6:
            patterns.append("Model Risk Above Threshold")

        return list(dict.fromkeys(patterns))

    def _get_historical_context(self, input_data: Dict) -> str:
        customer = input_data.get("customer_name", "Unknown")
        target_date = input_data.get("current_order_date", None)

        row = self._get_customer_day_row(customer, target_date)

        if row is None:
            return (
                f"Daily Velocity Alert Metrics:\n"
                f"- Customer Target: {customer}\n"
                f"- Evaluation Date: {target_date}\n"
                f"- No matching customer-day row found in customer_day_metrics"
            )

        return (
            f"Daily Velocity Alert Metrics:\n"
            f"- Customer Target: {row.get('customer_name', 'Unknown')}\n"
            f"- Evaluation Date: {pd.to_datetime(row.get('order_date')).date() if pd.notna(row.get('order_date')) else 'Unknown'}\n"
            f"- Daily Orders: {int(row.get('daily_orders', 0))}\n"
            f"- Weekly Orders: {float(row.get('weekly_orders', 0)):.2f}\n"
            f"- Monthly Orders: {float(row.get('monthly_orders', 0)):.2f}\n"
            f"- Spike Ratio: {float(row.get('spike_ratio', 0)):.2f}\n"
            f"- Velocity Alert Flag: {int(row.get('velocity_alert_flag', 0))}"
        )

    def analyze_abuse_case(self, input_data: Dict) -> Dict[str, Any]:
        metrics_row = self._get_customer_day_row(
            input_data.get("customer_name", "Unknown"),
            input_data.get("current_order_date")
        )

        abuse_score = 0.0
        if metrics_row is not None:
            abuse_score = (
                0.30 * min(float(metrics_row.get("daily_orders", 0)) / 10.0, 1.0) +
                0.20 * min(float(metrics_row.get("weekly_orders", 0)) / 30.0, 1.0) +
                0.20 * min(float(metrics_row.get("monthly_orders", 0)) / 90.0, 1.0) +
                0.20 * min(float(metrics_row.get("spike_ratio", 0)) / 3.0, 1.0) +
                0.10 * int(metrics_row.get("velocity_alert_flag", 0))
            )

        abuse_score = float(np.clip(abuse_score, 0, 1))
        risk_level = self._categorize_risk(abuse_score)
        patterns = self._analyze_pattern(metrics_row, abuse_score)
        historical_context = self._get_historical_context(input_data)

        prompt = (
            "You are a security analyst for a supermarket transaction platform.\n\n"
            f"{historical_context}\n\n"
            f"Risk level: {risk_level}\n"
            f"Abuse score: {abuse_score:.2f}\n"
            f"Patterns: {', '.join(patterns) if patterns else 'None'}\n\n"
            "Return a short production-ready assessment with:\n"
            "1. Likely issue\n"
            "2. Immediate action\n"
            "3. Follow-up action"
        )

        llm_analysis = self._call_llm_with_fallback(prompt)

        return {
            "customer_name": input_data.get("customer_name", "Unknown"),
            "current_order_date": input_data.get("current_order_date"),
            "abuse_score": abuse_score,
            "risk_level": risk_level,
            "patterns": patterns,
            "historical_context": historical_context,
            "llm_analysis": llm_analysis,
            "metrics_row": metrics_row.to_dict() if metrics_row is not None else None,
        }

## Unit test LLM

In [31]:
import random

customer_day_metrics = pd.read_parquet(CUSTOMER_METRICS_PATH)
customer_day_metrics['order_date'] = pd.to_datetime(customer_day_metrics['order_date'], errors='coerce')

# Call agent
agent = AbuseInvestigationAgent(
    hf_llm=hf_model,
    groq_llm=groq_model,
    historical_df=customer_day_metrics
)

# Clean and sample len records FIRST using pandas
cleaned_df = customer_day_metrics.dropna(subset=["customer_name", "order_date"])
sampled_df = cleaned_df.sample(n=150, random_state=42)

# Iterrows only runs as len times needed
test_inputs = [
    {
        "customer_name": row["customer_name"],
        "current_order_date": row["order_date"].strftime("%Y-%m-%d")
    }
    for _, row in sampled_df.iterrows()
]

In [32]:
# Save the results to a JSON file
import json
from pathlib import Path

output_path = BASE_DIR / "app" / "data" / "abuse_detection_json" / "abuse_analysis_results_2.json"

all_results = []

for test_input in test_inputs:
    result = agent.analyze_abuse_case(test_input)

    llm_text = result["llm_analysis"]
    if hasattr(llm_text, "content"):
        llm_text = llm_text.content

    all_results.append({
        "customer_name": result["customer_name"],
        "current_order_date": result["current_order_date"],
        "abuse_score": result["abuse_score"],
        "risk_level": result["risk_level"],
        "patterns": result["patterns"],
        "historical_context": result["historical_context"],
        "llm_analysis": llm_text,
        "metrics_row": result["metrics_row"],
    })

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False, default=str)

print(f"Saved {len(all_results)} results to {output_path}")

Saved 150 results to /Users/miftahhadiyannoor/Documents/Supermarket_Sales/Production/app/data/abuse_detection_json/abuse_analysis_results_2.json
